This notebook contains a final retraining of Catboost, XGBoost and LightGBM for the residential datasets using the optimised parameters obtained from optuna. Refer to optimisation.ipynb and extract_best_parameters.ipynb for more information:


### Importing libraries and loading files:

In [7]:
import os
import json
import joblib
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from catboost import CatBoostRegressor
import xgboost as xgb
import lightgbm as lgb
from pathlib import Path




In [8]:
train_global=pd.read_csv(r"datasets\train_global.csv")
test_global=pd.read_csv(r"datasets\test_global.csv")
train_house=pd.read_csv(r"datasets\train_house.csv")
test_house=pd.read_csv(r"datasets\test_house.csv")
train_apartment=pd.read_csv(r"datasets\train_apartment.csv")
test_apartment=pd.read_csv(r"datasets\test_apartment.csv")
train_luxury=pd.read_csv(r"datasets\train_luxury.csv")
test_luxury=pd.read_csv(r"datasets\test_luxury.csv")
train_luxury_apartment=pd.read_csv(r"datasets\train_luxury_apartment.csv")
test_luxury_apartment=pd.read_csv(r"datasets\test_luxury_apartment.csv")
train_luxury_house=pd.read_csv(r"datasets\train_luxury_house.csv")
test_luxury_house=pd.read_csv(r"datasets\test_luxury_house.csv")
train_residential_apartment=pd.read_csv(r"datasets\train_residential_apartment.csv")
test_residential_apartment=pd.read_csv(r"datasets\test_residential_apartment.csv")
train_residential_house=pd.read_csv(r"datasets\train_residential_house.csv")
test_residential_house=pd.read_csv(r"datasets\test_residential_house.csv")
train_residential=pd.read_csv(r"datasets\train_residential.csv")
test_residential=pd.read_csv(r"datasets\test_residential.csv")

### Retraining models:


In [ ]:

datasets = {
    "res_apt": (train_residential_apartment, test_residential_apartment),
    "res_house": (train_residential_house, test_residential_house)
}


def prepare_data(train_df, test_df):

    train_df = train_df.copy()
    test_df = test_df.copy()

    
    train_df = train_df[train_df["price"] > 0]
    test_df = test_df[test_df["price"] > 0]

    
    train_df["log_price"] = np.log(train_df["price"])
    test_df["log_price"] = np.log(test_df["price"])

    # target
    y_train = train_df["log_price"]
    y_test = test_df["log_price"]

    # input features
    X_train = train_df.drop(columns=["price", "log_price"])
    X_test = test_df.drop(columns=["price", "log_price"])

    return X_train, y_train, X_test, y_test, train_df.columns.tolist()



def build_preprocessor(X):
    num_cols = X.select_dtypes(include=["int64", "float64"]).columns
    cat_cols = X.select_dtypes(include=["object"]).columns

    pre = ColumnTransformer([

        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols),

        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols)

    ])

    return pre



def load_params(json_path):
    with open(json_path, "r") as f:
        return json.load(f)




def build_model(model_name, params):

    if model_name == "cat":
        allowed = {
            "depth", "learning_rate", "l2_leaf_reg",
            "bagging_temperature", "random_strength",
            "iterations", "loss_function", "verbose"
        }
        filtered = {k: v for k, v in params.items() if k in allowed}
        return CatBoostRegressor(**filtered)

    if model_name == "xgb":
        allowed = {
            "max_depth", "learning_rate", "subsample",
            "colsample_bytree", "min_child_weight",
            "n_estimators", "objective", "tree_method"
        }
        filtered = {k: v for k, v in params.items() if k in allowed}
        return xgb.XGBRegressor(**filtered)

    if model_name == "lgbm":
        allowed = {
            "learning_rate", "num_leaves", "feature_fraction",
            "bagging_fraction", "bagging_freq", "min_data_in_leaf",
            "objective", "metric"
        }
        filtered = {k: v for k, v in params.items() if k in allowed}
        return lgb.LGBMRegressor(**filtered)

    raise ValueError("Unknown model: " + model_name)





def evaluate(model, X_train, y_train, X_test, y_test):

    # TRAIN
    pred_train_log = model.predict(X_train)
    pred_train = np.exp(pred_train_log)

    rmse_train = float(np.sqrt(mean_squared_error(np.exp(y_train), pred_train)))
    mae_train = float(mean_absolute_error(np.exp(y_train), pred_train))
    r2_train = float(r2_score(np.exp(y_train), pred_train))

    # TEST
    pred_test_log = model.predict(X_test)
    pred_test = np.exp(pred_test_log)

    rmse_test = float(np.sqrt(mean_squared_error(np.exp(y_test), pred_test)))
    mae_test = float(mean_absolute_error(np.exp(y_test), pred_test))
    r2_test = float(r2_score(np.exp(y_test), pred_test))

    return rmse_train, mae_train, r2_train, rmse_test, mae_test, r2_test



MODEL_CONFIGS = [
    ("cat",  r"models_optimisation\best_parameters\cat_res_apt_best_params.json"),
    ("cat",  r"models_optimisation\best_parameters\cat_res_house_best_params.json"),
    ("lgbm",  r"models_optimisation\best_parameters\lgbm_res_apt_best_params.json"),
    ("lgbm",  r"models_optimisation\best_parameters\lgbm_res_house_best_params.json"),
    ("xgb", r"models_optimisation\best_parameters\xgb_res_apt_best_params.json"),
    ("xgb", r"models_optimisation\best_parameters\xgb_res_house_best_params.json")


]






results = []
os.makedirs(r"models_optimisation/optimum_models", exist_ok=True)
os.makedirs(r"models_optimisation/plots_overfit", exist_ok=True)

for model_name, json_file in MODEL_CONFIGS:

    
    json_name = Path(json_file).name  # for example "cat_res_apt_best_params.json"
    dataset_name = json_name.replace(f"{model_name}_", "").replace("_best_params.json", "")
    print("Resolved dataset_name =", dataset_name)


    if dataset_name not in datasets:
        print(f"[WARN] Dataset {dataset_name} not found, skipping")
        continue

    print(f"\n=== Training {model_name.upper()} on dataset: {dataset_name} ===")

    train_df, test_df = datasets[dataset_name]

    X_train, y_train, X_test, y_test, cols = prepare_data(train_df, test_df)

    pre = build_preprocessor(X_train)
    params = load_params(json_file)

    model = build_model(model_name, params)

    full_model = Pipeline([
        ("preprocess", pre),
        ("model", model)
    ])

    full_model.fit(X_train, y_train)

    # --- evaluation
    rmse_train, mae_train, r2_train, rmse_test, mae_test, r2_test = evaluate(
        full_model, X_train, y_train, X_test, y_test
    )

    # save
    save_path = f"models_optimisation/optimum_models/model_{model_name}_{dataset_name}.pkl"
    joblib.dump(full_model, save_path)

    results.append({
        "model": model_name,
        "dataset": dataset_name,
        "rmse_train": rmse_train,
        "mae_train": mae_train,
        "r2_train": r2_train,
        "rmse_test": rmse_test,
        "mae_test": mae_test,
        "r2_test": r2_test,
        "model_path": save_path
    })

    # plot
    plt.figure(figsize=(5,4))
    plt.bar(["train", "test"], [rmse_train, rmse_test], color=["steelblue", "firebrick"])
    plt.title(f"{model_name.upper()} – {dataset_name} (RMSE)")
    plt.ylabel("RMSE")
    plt.tight_layout()
    plt.savefig(f"models_optimisation/plots_overfit/{model_name}_{dataset_name}.png")
    plt.close()

# ======================================================
# 9) EXCEL / CSV REPORT
# ======================================================

df = pd.DataFrame(results)
df.to_csv(r"models_optimisation/trained_models_summary.csv", index=False)
print("\nSaved trained_models_summary.csv")

print("\n=== DONE ===")


Resolved dataset_name = res_apt

=== Training CAT on dataset: res_apt ===
0:	learn: 0.4181808	total: 10.6ms	remaining: 50.3s
1:	learn: 0.4054495	total: 22.4ms	remaining: 53.1s
2:	learn: 0.3918381	total: 34.2ms	remaining: 54s
3:	learn: 0.3793695	total: 46.9ms	remaining: 55.6s
4:	learn: 0.3680151	total: 59.1ms	remaining: 56s
5:	learn: 0.3589706	total: 74.7ms	remaining: 58.9s
6:	learn: 0.3507933	total: 87.9ms	remaining: 59.4s
7:	learn: 0.3436617	total: 102ms	remaining: 1m
8:	learn: 0.3369498	total: 116ms	remaining: 1m
9:	learn: 0.3309172	total: 130ms	remaining: 1m 1s
10:	learn: 0.3256141	total: 144ms	remaining: 1m 1s
11:	learn: 0.3206006	total: 157ms	remaining: 1m 1s
12:	learn: 0.3172665	total: 170ms	remaining: 1m 1s
13:	learn: 0.3132769	total: 184ms	remaining: 1m 2s
14:	learn: 0.3104425	total: 205ms	remaining: 1m 4s
15:	learn: 0.3071047	total: 221ms	remaining: 1m 5s
16:	learn: 0.3041902	total: 238ms	remaining: 1m 6s
17:	learn: 0.3015319	total: 257ms	remaining: 1m 7s
18:	learn: 0.2991610	

c:\Users\wporc\GitHub\Immo-Eliza-Machine-Learning\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\wporc\GitHub\Immo-Eliza-Machine-Learning\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] min_data_in_leaf is set=18, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=18
[LightGBM] [Warning] feature_fraction is set=0.9973538845465474, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9973538845465474
[LightGBM] [Warning] bagging_fraction is set=0.8368796741478572, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8368796741478572
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6
[LightGBM] [Warning] min_data_in_leaf is set=18, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=18
[LightGBM] [Warning] feature_fraction is set=0.9973538845465474, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9973538845465474
[LightGBM] [Warning] bagging_fraction is set=0.8368796741478572, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8368796741478572
[LightGBM] [Warning] bagging_freq is set=6, su

c:\Users\wporc\GitHub\Immo-Eliza-Machine-Learning\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\wporc\GitHub\Immo-Eliza-Machine-Learning\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] min_data_in_leaf is set=16, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=16
[LightGBM] [Warning] feature_fraction is set=0.8644735870963574, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8644735870963574
[LightGBM] [Warning] bagging_fraction is set=0.9976568288314043, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9976568288314043
[LightGBM] [Warning] bagging_freq is set=7, subsample_freq=0 will be ignored. Current value: bagging_freq=7
[LightGBM] [Warning] min_data_in_leaf is set=16, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=16
[LightGBM] [Warning] feature_fraction is set=0.8644735870963574, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8644735870963574
[LightGBM] [Warning] bagging_fraction is set=0.9976568288314043, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9976568288314043
[LightGBM] [Warning] bagging_freq is set=7, su

The models are optimised and can be now used to create a predict.py script for later deployment